# RQ3 Main Analysis: Social Determinants of Electricity Sufficiency

Clean main-analysis notebook for the final RQ3 specification. The primary model uses the reduced M4 specification and the log electricity sufficiency ratio as the outcome.

Primary specification:

`log(sufficiency_ratio) ~ SES + rurality + education + female share + log(GDP per capita) + log(population) + total HDD + total CDD + country fixed effects`

Standard errors are clustered by country. Predictors are standardized, so coefficients represent a one-standard-deviation increase. Percent-change effects are reported as `100 * (exp(beta) - 1)`.

Interactions are estimated only as pre-specified exploratory checks. Because the interaction family does not survive FDR correction, interaction results are not treated as confirmatory headline findings.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
RESULTS_DIR = PROJECT / 'Results/RQ3_social_determinants/Main'
TABLE_DIR = RESULTS_DIR / 'Tables'
FIG_DIR = RESULTS_DIR / 'Figures'
PROCESS_DIR = PROJECT / 'Process/RQ3_main_analysis_clean'
for d in [RESULTS_DIR, TABLE_DIR, FIG_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Data:', DATA_PATH)
print('Main results:', RESULTS_DIR)


In [ ]:

def star(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.1: return '†'
    return ''

def pct_change(beta):
    return (np.exp(beta) - 1) * 100

def clean_model_data(df):
    d = df.copy()
    d['income_clean'] = d['income'].where(d['income'] >= 0, np.nan)
    d['wealth_clean'] = d['wealth'].where(d['wealth'] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

raw = pd.read_csv(DATA_PATH)
df = clean_model_data(raw)

social_predictors = ['SES_raw', 'rural', 'education', 'female_share']
controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
M4_predictors = social_predictors + controls
required = ['GDLcode', 'iso_code', 'country', 'analytical_region', 'population', 'log_sufficiency_ratio'] + M4_predictors
model_data = df.dropna(subset=required).copy()
model_data = model_data[(model_data['population'] > 0) & np.isfinite(model_data['log_sufficiency_ratio'])].copy()

std = model_data.copy()
standardization=[]
for var in M4_predictors + ['age', 'Hhsize']:
    if var in std.columns:
        mu = std[var].mean()
        sd = std[var].std(ddof=0)
        std['z_' + var] = (std[var] - mu) / sd
        standardization.append({'variable': var, 'mean': float(mu), 'sd': float(sd)})
pd.DataFrame(standardization).to_csv(PROCESS_DIR / 'RQ3_main_standardization_parameters.csv', index=False)

sample_summary = pd.DataFrame([
    {'metric': 'GDL regions', 'value': len(std)},
    {'metric': 'countries', 'value': std['iso_code'].nunique()},
    {'metric': 'analytical regions', 'value': std['analytical_region'].nunique()},
])
sample_summary.to_csv(TABLE_DIR / 'Table_RQ3_main_sample_summary.csv', index=False)
print(sample_summary)
print(std.groupby('analytical_region').size())


In [ ]:

def fit_fe_ols(data, predictors, outcome='log_sufficiency_ratio'):
    rhs = ' + '.join(['z_' + v for v in predictors]) + ' + C(iso_code)'
    formula = outcome + ' ~ ' + rhs
    return smf.ols(formula, data=data).fit(cov_type='cluster', cov_kwds={'groups': data['iso_code']})

def extract_ols(fit, predictors, scope, region=None):
    ci = fit.conf_int()
    rows=[]
    for var in predictors:
        param='z_' + var
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        rows.append({
            'scope': scope,
            'region': region,
            'variable': var,
            'coef_log_points_per_sd': b,
            'cluster_se': se,
            'ci_low': float(lo),
            'ci_high': float(hi),
            'p_value': p,
            'stars': star(p),
            'pct_change_per_sd': pct_change(b),
            'pct_ci_low': pct_change(lo),
            'pct_ci_high': pct_change(hi),
            'n_obs': int(fit.nobs),
            'n_countries': int(data_for_fit['iso_code'].nunique()) if False else np.nan,
            'r_squared': float(fit.rsquared),
            'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic),
            'bic': float(fit.bic),
        })
    return rows

# Global M4 main model.
global_fit = fit_fe_ols(std, M4_predictors)
global_rows=[]
ci = global_fit.conf_int()
for var in M4_predictors:
    param='z_' + var
    b=float(global_fit.params[param]); se=float(global_fit.bse[param]); p=float(global_fit.pvalues[param])
    lo, hi = ci.loc[param].tolist()
    global_rows.append({
        'model': 'M4_main_country_FE_clustered',
        'outcome': 'log_sufficiency_ratio',
        'variable': var,
        'coef_log_points_per_sd': b,
        'cluster_se': se,
        'ci_low': float(lo),
        'ci_high': float(hi),
        'p_value': p,
        'stars': star(p),
        'pct_change_per_sd': pct_change(b),
        'pct_ci_low': pct_change(lo),
        'pct_ci_high': pct_change(hi),
        'n_obs': int(global_fit.nobs),
        'n_countries': int(std['iso_code'].nunique()),
        'r_squared': float(global_fit.rsquared),
        'adj_r_squared': float(global_fit.rsquared_adj),
        'aic': float(global_fit.aic),
        'bic': float(global_fit.bic),
    })

global_table = pd.DataFrame(global_rows)
global_table.to_csv(TABLE_DIR / 'Table_RQ3_main_global_M4_coefficients.csv', index=False)
model_summary = pd.DataFrame([{
    'model': 'M4_main_country_FE_clustered',
    'outcome': 'log_sufficiency_ratio',
    'n_obs': int(global_fit.nobs),
    'n_countries': int(std['iso_code'].nunique()),
    'r_squared': float(global_fit.rsquared),
    'adj_r_squared': float(global_fit.rsquared_adj),
    'aic': float(global_fit.aic),
    'bic': float(global_fit.bic),
}])
model_summary.to_csv(TABLE_DIR / 'Table_RQ3_main_global_M4_model_summary.csv', index=False)
global_table


In [ ]:

# Region-stratified M4 models.
region_rows=[]
region_summary=[]
for region, dreg in std.groupby('analytical_region', dropna=False):
    dreg = dreg.copy()
    n_countries = dreg['iso_code'].nunique()
    n_obs = len(dreg)
    if n_countries < 3 or n_obs < 30:
        region_summary.append({'region': region, 'n_obs': n_obs, 'n_countries': n_countries, 'status': 'skipped'})
        continue
    fit = fit_fe_ols(dreg, M4_predictors)
    ci = fit.conf_int()
    for var in M4_predictors:
        param='z_' + var
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        region_rows.append({
            'model': 'M4_region_country_FE_clustered',
            'outcome': 'log_sufficiency_ratio',
            'region': region,
            'variable': var,
            'coef_log_points_per_sd': b,
            'cluster_se': se,
            'ci_low': float(lo),
            'ci_high': float(hi),
            'p_value': p,
            'stars': star(p),
            'pct_change_per_sd': pct_change(b),
            'pct_ci_low': pct_change(lo),
            'pct_ci_high': pct_change(hi),
            'n_obs': int(fit.nobs),
            'n_countries': int(n_countries),
            'r_squared': float(fit.rsquared),
            'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic),
            'bic': float(fit.bic),
        })
    region_summary.append({
        'region': region, 'n_obs': n_obs, 'n_countries': n_countries, 'status': 'ok',
        'r_squared': float(fit.rsquared), 'adj_r_squared': float(fit.rsquared_adj),
        'aic': float(fit.aic), 'bic': float(fit.bic)
    })

region_table = pd.DataFrame(region_rows)
region_summary = pd.DataFrame(region_summary)
region_table.to_csv(TABLE_DIR / 'Table_RQ3_main_regional_M4_coefficients.csv', index=False)
region_summary.to_csv(TABLE_DIR / 'Table_RQ3_main_regional_M4_model_summary.csv', index=False)
region_table[region_table['variable'].isin(['female_share','education'])].sort_values(['variable','p_value'])


In [ ]:

# Pre-specified M4 interactions. These are retained as exploratory because FDR does not reject the family of tests.
interaction_pairs = [
    ('rural', 'SES_raw', 'Rurality x SES'),
    ('rural', 'education', 'Rurality x Education'),
    ('female_share', 'SES_raw', 'Female share x SES'),
    ('female_share', 'education', 'Female share x Education'),
    ('SES_raw', 'education', 'SES x Education'),
    ('age', 'female_share', 'Age x Female share'),
    ('SES_raw', 'Hhsize', 'SES x Household size'),
    ('female_share', 'rural', 'Female share x Rurality'),
]

interaction_rows=[]
slope_rows=[]
for x, z, label in interaction_pairs:
    predictors = list(dict.fromkeys(M4_predictors + [x, z]))
    needed = ['log_sufficiency_ratio', 'iso_code'] + ['z_' + v for v in predictors]
    d = std.dropna(subset=needed).copy()
    term = f'z_{x}:z_{z}'
    rhs = ' + '.join(['z_' + v for v in predictors] + [term, 'C(iso_code)'])
    fit = smf.ols('log_sufficiency_ratio ~ ' + rhs, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['iso_code']})
    ci = fit.conf_int()
    b=float(fit.params[term]); se=float(fit.bse[term]); p=float(fit.pvalues[term])
    lo, hi = ci.loc[term].tolist()
    interaction_rows.append({
        'model': 'M4_targeted_interaction_country_FE_clustered',
        'outcome': 'log_sufficiency_ratio',
        'interaction': label,
        'x': x,
        'moderator': z,
        'coef_interaction_log_points': b,
        'cluster_se': se,
        'ci_low': float(lo),
        'ci_high': float(hi),
        'p_value': p,
        'stars_raw': star(p),
        'n_obs': int(fit.nobs),
        'n_countries': int(d['iso_code'].nunique()),
        'included_predictors': ', '.join(predictors),
    })
    cov = fit.cov_params()
    for focal, mod in [(x, z), (z, x)]:
        focal_term = 'z_' + focal
        int_term = term
        for level_name, mod_value in [('low_-1SD', -1.0), ('mean_0SD', 0.0), ('high_+1SD', 1.0)]:
            slope = float(fit.params[focal_term] + fit.params[int_term] * mod_value)
            var_slope = float(cov.loc[focal_term, focal_term] + (mod_value ** 2) * cov.loc[int_term, int_term] + 2 * mod_value * cov.loc[focal_term, int_term])
            se_slope = np.sqrt(max(var_slope, 0))
            z_stat = slope / se_slope if se_slope > 0 else np.nan
            p_slope = 2 * (1 - stats.norm.cdf(abs(z_stat))) if np.isfinite(z_stat) else np.nan
            slope_rows.append({
                'model': 'M4_targeted_interaction_country_FE_clustered',
                'outcome': 'log_sufficiency_ratio',
                'interaction': label,
                'focal_variable': focal,
                'moderator_variable': mod,
                'moderator_level': level_name,
                'moderator_z_value': mod_value,
                'slope_log_points_per_sd': slope,
                'se': se_slope,
                'ci_low': slope - 1.96 * se_slope,
                'ci_high': slope + 1.96 * se_slope,
                'p_value': p_slope,
                'stars': star(p_slope),
                'pct_change_per_sd': pct_change(slope),
            })

interaction_table = pd.DataFrame(interaction_rows)
rejected, qvals = fdrcorrection(interaction_table['p_value'].values, alpha=0.05)
interaction_table['q_value_fdr'] = qvals
interaction_table['fdr_reject_0_05'] = rejected
interaction_table['stars_fdr'] = interaction_table['q_value_fdr'].apply(star)
slope_table = pd.DataFrame(slope_rows)

focused = interaction_table[interaction_table['interaction'].isin(['Female share x Education', 'Age x Female share'])].copy()
focused_slopes = slope_table[
    (slope_table['interaction'].isin(['Female share x Education', 'Age x Female share'])) &
    (slope_table['focal_variable'].eq('female_share'))
].copy()

interaction_table.to_csv(TABLE_DIR / 'Table_RQ3_main_exploratory_interactions_M4_all.csv', index=False)
focused.to_csv(TABLE_DIR / 'Table_RQ3_main_focused_interactions_M4.csv', index=False)
focused_slopes.to_csv(TABLE_DIR / 'Table_RQ3_main_focused_interaction_simple_slopes_M4.csv', index=False)
interaction_table.sort_values('p_value')


In [ ]:

# Figure 1: global M4 coefficients.
labels = {
    'SES_raw': 'SES', 'rural': 'Rurality', 'education': 'Education', 'female_share': 'Female share',
    'log_GDP_pc': 'GDP per capita', 'log_population': 'Population', 'total_HDD': 'Heating degree days', 'total_CDD': 'Cooling degree days'
}
plot = global_table.copy()
plot['label'] = plot['variable'].map(labels)
plot = plot.iloc[::-1]
colors = ['#4c78a8' if v in social_predictors else '#8a8f98' for v in plot['variable']]
fig, ax = plt.subplots(figsize=(7.0, 4.8), constrained_layout=True)
ax.errorbar(plot['pct_change_per_sd'], plot['label'],
            xerr=[plot['pct_change_per_sd'] - plot['pct_ci_low'], plot['pct_ci_high'] - plot['pct_change_per_sd']],
            fmt='none', ecolor='#4a4a4a', elinewidth=1.2, capsize=3, zorder=1)
ax.scatter(plot['pct_change_per_sd'], plot['label'], s=60, color=colors, zorder=2)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_xlabel('Percent change in sufficiency ratio per 1 SD increase')
ax.set_ylabel('')
ax.set_title('Global social determinants of electricity sufficiency')
for _, row in plot.iterrows():
    if row['stars']:
        ax.text(row['pct_ci_high'] + 1.5, row['label'], row['stars'], va='center', fontsize=10)
fig.savefig(FIG_DIR / 'Figure_RQ3_main_global_M4_coefficients.png', bbox_inches='tight')
plt.show()


In [ ]:

# Figure 2: regional social coefficients for M4.
social = region_table[region_table['variable'].isin(social_predictors)].copy()
social['label'] = social['variable'].map(labels)
region_order = region_summary.sort_values('region')['region'].tolist()
var_order = ['SES_raw', 'rural', 'education', 'female_share']
fig, axes = plt.subplots(1, 4, figsize=(13.5, 5.2), sharey=True, constrained_layout=True)
for ax, var in zip(axes, var_order):
    d = social[social['variable'].eq(var)].copy()
    d['region'] = pd.Categorical(d['region'], categories=region_order, ordered=True)
    d = d.sort_values('region')
    y = np.arange(len(d))
    ax.errorbar(d['pct_change_per_sd'], y,
                xerr=[d['pct_change_per_sd'] - d['pct_ci_low'], d['pct_ci_high'] - d['pct_change_per_sd']],
                fmt='none', ecolor='#4a4a4a', elinewidth=1.0, capsize=2)
    ax.scatter(d['pct_change_per_sd'], y, s=42, color='#2f6f73')
    ax.axvline(0, color='#333333', linewidth=0.8)
    ax.set_title(labels[var])
    ax.set_xlabel('% change')
    ax.set_yticks(y)
    ax.set_yticklabels(d['region'] if ax is axes[0] else [])
    for yy, (_, row) in enumerate(d.iterrows()):
        if row['stars']:
            ax.text(row['pct_ci_high'] + 2, yy, row['stars'], va='center', fontsize=9)
axes[0].set_ylabel('Analytical region')
fig.suptitle('Regional heterogeneity in M4 social determinants', y=1.03)
fig.savefig(FIG_DIR / 'Figure_RQ3_main_regional_social_coefficients_M4.png', bbox_inches='tight')
plt.show()


In [ ]:

# Figure 3: focused exploratory interaction simple slopes.
if not focused_slopes.empty:
    fp = focused_slopes.copy()
    fp['moderator_level_label'] = fp['moderator_level'].map({'low_-1SD':'Low (-1 SD)', 'mean_0SD':'Mean', 'high_+1SD':'High (+1 SD)'})
    fp['interaction_label'] = fp['interaction'].map({'Female share x Education':'Female share slope by education', 'Age x Female share':'Female share slope by age'})
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6), sharey=False, constrained_layout=True)
    for ax, (name, d) in zip(axes, fp.groupby('interaction_label', sort=False)):
        d = d.copy()
        x = np.arange(len(d))
        ax.errorbar(x, d['pct_change_per_sd'],
                    yerr=[d['pct_change_per_sd'] - d['ci_low'].map(pct_change), d['ci_high'].map(pct_change) - d['pct_change_per_sd']],
                    fmt='none', ecolor='#4a4a4a', elinewidth=1.0, capsize=3)
        ax.scatter(x, d['pct_change_per_sd'], s=50, color='#7a5195')
        ax.axhline(0, color='#333333', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(d['moderator_level_label'], rotation=0)
        ax.set_title(name)
        ax.set_ylabel('% change in sufficiency ratio per 1 SD female share')
        for xx, (_, row) in enumerate(d.iterrows()):
            if row['stars']:
                ax.text(xx, row['pct_change_per_sd'] + 6, row['stars'], ha='center', fontsize=9)
    fig.suptitle('Exploratory interaction patterns, M4', y=1.04)
    fig.savefig(FIG_DIR / 'Figure_RQ3_main_focused_interaction_simple_slopes_M4.png', bbox_inches='tight')
    plt.show()


In [ ]:

manifest = {
    'notebook': str(PROJECT / 'RQ3_main_analysis_clean.ipynb'),
    'data': str(DATA_PATH),
    'results_dir': str(RESULTS_DIR),
    'primary_model': 'M4: log_sufficiency_ratio with country FE and country-clustered SE',
    'interaction_treatment': 'estimated as pre-specified exploratory checks; not confirmatory because FDR q-values do not reject the interaction family',
    'sample_n': int(len(std)),
    'countries': int(std['iso_code'].nunique()),
    'main_tables': sorted([str(p) for p in TABLE_DIR.glob('*.csv')]),
    'main_figures': sorted([str(p) for p in FIG_DIR.glob('*.png')]),
}
(PROCESS_DIR / 'RQ3_main_analysis_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
